# TP Integrador Neo4j, MongoDB y Redis
Ghizzo Agustín
Meza Joaquín

In [ ]:
# Celda de importaciones y conexiones
import os 
import redis 
from pymongo import MongoClient 
from neo4j import GraphDatabase

# Conexiones
MONGO_HOST = "mongo"
REDIS_HOST = "redis"
NEO4J_HOST = "neo4j"

MONGO_USER = os.getenv("MONGO_INITDB_ROOT_USERNAME", "admin")
MONGO_PASS = os.getenv("MONGO_INITDB_ROOT_PASSWORD", "admin123")
REDIS_PASS = os.getenv("REDIS_PASSWORD", "redis123")
NEO4J_PASS = os.getenv("NEO4J_PASSWORD", "neo4j123")

MONGO_URI = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:27017/"
NEO4J_URI = f"bolt://{NEO4J_HOST}:7687"

# Inicialización de conexiones 

try:
    mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    db_mongo = mongo_client["tpIntegrador"]
    mongo_client.admin.command('ping')
    print("Conexión a MongoDB exitosa")

    redis_client = redis.Redis(host=REDIS_HOST, port=6379, password=REDIS_PASS, decode_responses=True)
    redis_client.ping()
    print("Conexión exitosa a Redis")

    neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=("neo4j", NEO4J_PASS))
    with neo4j_driver.session() as session:
        session.run("RETURN 1")
    print("Conexión a Neo4j exitosa")

except Exception as e:
    print({e})


# Se instancia fecha referencia en formato ISO

FECHA_REFERENCIA = "2026-10-01"

# Carga de las bases de datos




In [ ]:
#celda provisoria de importaciones
import sys
import os

sys.path.append(os.path.abspath("../python"))
sys.path.append("/work/python")

# Inciso 3 - 12 consultas

In [ ]:
import neo4j
from neo4j._async import home_db_cache
from neo4j._async import home_db_cache
from dns.rdtypes.ANY import CDNSKEY
from dns.rdtypes.ANY import CDNSKEY

# provisorio:

from python.carga_inicial import insert_mongo, insert_neo4j, insert_redis


#Limpio las bases de datos 

for col in ["usuarios", "artistas", "lugares", "eventos", "compras"]:
    db_mongo[col].delete_many({})

redis_client.flushdb()

with neo4j_driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")

print("Bases de datos limpias")

#Carga de bases de datos

insert_mongo(db_mongo)
insert_redis(redis_client)
insert_neo4j(neo4j_driver)



# 1) Usuarios que asistieron a algún evento de "Los Cóndores Eléctricos"
print("Inciso 1: ")

artista = db_mongo.artistas.find_one({"nombre":"Los Cóndores Eléctricos"})
artista_id = artista["artista_id"] #es uno?

eventos_pasados = db_mongo.eventos.find({"artistas":artista_id, "fecha":{"$lt":FECHA_REFERENCIA} })
eventos_ids = [e["evento_id"] for e in eventos_pasados]

compras_confirmadas = db_mongo.compras.find({"evento_id": {"$in": eventos_ids }, "estado": "confirmada"})

#Quitando duplicados
usuarios_ids = list(set(c["usuario_id"] for c in compras_confirmadas))

usuarios = db_mongo.usuarios.find({"usuario_id": {"$in": usuarios_ids}})

#Imprimiendo

print("Usuarios que asistieron a al menos un evento de 'Los cóndores Eléctricos': ")
for usu in usuarios:
    print(f" {usu['nombre']} (ID: {usu['usuario_id']})")

# 2) Amigos de Juan López que siguen algún artista que también sigue Juan. Mostrar el nombre del amigo y el artista.
print()
print("Inciso 2: Amigos de Juan")
query_2 = """
MATCH (juan:Usuario {nombre: "Juan López"})-[:AMIGO_DE]-(amigo:Usuario), (juan)-[:SIGUE]->(artista:Artista), (amigo)-[:SIGUE]->(artista)
RETURN amigo.nombre, artista.nombre
"""

with neo4j_driver.session() as session:
    for r in session.run(query_2):
        print(f"Con el amigo: {r['amigo.nombre']} comparten al artista: {r['artista.nombre']}")

# 3) Recomendar a Juan artistas que siguen sus amigos y que él no sigue, ordenados por cantidad de amigos que los siguen.
print()
print("Inciso 3: ")
query_3= """
MATCH (juan:Usuario {nombre: "Juan López"})-[:AMIGO_DE]-(amigo:Usuario)-[:SIGUE]->(artista:Artista)
WITH juan, artista, amigo, EXISTS((juan)-[:SIGUE]->(artista)) AS juan_ya_lo_sigue
WHERE NOT juan_ya_lo_sigue
RETURN artista.nombre, COUNT(amigo) AS cantidad_amigos
ORDER BY cantidad_amigos DESC
"""

with neo4j_driver.session() as session:
    for r in session.run(query_3):
        print(f"Artista recomendado: {r['artista.nombre']} , seguido por {r['cantidad_amigos']} de tus amigos")

# 4) Para los artistas del punto anterior, listar sus eventos futuros con lugar, ciudad, fecha, precio y entradas disponibles.
print()
print("Inciso 4: ")

with neo4j_driver.session() as session:
    nombres_recomendados = [r['artista.nombre'] for r in session.run(query_3)]

artistas_mongo = db_mongo.artistas.find({"nombre": {"$in": nombres_recomendados}})
ids_recomendados = [artista['artista_id'] for artista in artistas_mongo]

eventos_futuros_recomendados = db_mongo.eventos.find({
    'artistas':{'$in':ids_recomendados},
    'fecha':{'$gt':FECHA_REFERENCIA},
    'estado':'activo'
})

for e in eventos_futuros_recomendados:
    print(f"Evento recomendado: {e['nombre']} el {e['fecha']} a ${e['precio']}")


# 5) Artistas similares a los que sigue Lucía Romero y que ella no sigue.
print()
print("Inciso 5: ")
query_4= """
MATCH (lucia:Usuario {nombre: "Lucía Romero"})-[:SIGUE]->(artista:Artista)-[:SIMILAR]-(artistaSimilar:Artista)
WITH lucia,artista,artistaSimilar, EXISTS((lucia)-[:SIGUE]->(artista)) AS lucia_ya_lo_sigue
WHERE NOT lucia_ya_lo_sigue
RETURN artista.nombre
"""

with neo4j_driver.session() as session:
    artistas_recomendados = set(r['artista.nombre'] for r in session.run(query_3))

    print(f"Quizas te interesen: ", artistas_recomendados)



